# Notebook 2: MinHash Entity Resolution

## Overview

Federal procurement data exhibits a well-known entity resolution problem:
the same contractor can appear under multiple name variants depending on which
office or facility registered the contract. For example, across FY2025
definitive contracts, Lockheed Martin alone appears under 13 distinct name
strings including `LOCKHEED MARTIN CORPORATION`, `LOCKHEED MARTIN CORP`, and
`LOCKHEED MARTIN GLOBAL INC`. Without resolution, these would appear as
separate nodes in the network graph, fragmenting a single contractor's true
influence across multiple disconnected entities.

## Two-Stage Approach

Entity resolution is handled across two notebooks:

- **Stage 1 (Notebook 1):** Deterministic normalization: punctuation removal,
  ampersand standardization, whitespace collapse. Resolves exact string
  duplicates that differ only by punctuation.
- **Stage 2 (This notebook):** Probabilistic resolution via MinHash LSH
  identifies and merges near-duplicate names that survive Stage 1, such as
  abbreviation variants that remain distinct strings after normalization.

## Merging Philosophy

This notebook attempts to resolve names at the **registered entity level**. The goal is
to merge name strings that refer to the same SAM.gov registered contracting
entity, not to consolidate corporate families or parent companies.

Subsidiaries like `LOCKHEED MARTIN SERVICES LLC` and `LOCKHEED MARTIN
INTEGRATED SYSTEMS LLC` are legally distinct registered entities that win
contracts independently. Merging them here would collapse meaningful network
structure in the HITS graph, where each subsidiary's individual agency
relationships carry analytical weight.

A Jaccard similarity threshold of 0.85 on character-level 3-shingles is used,
with additional filters to block merges across numbered joint ventures,
geographic specializations, and conflicting legal suffixes. This conservative
threshold merges only clear abbreviation variants (e.g. `CORP` vs
`CORPORATION`) where there is high confidence both strings refer to the same
registered entity.

Note: Corporate family aggregation, the grouping of subsidiaries under a common
parent, is performed separately in Notebooks 3 and 5 as a deliberate
analytical step for scoring and clustering, after entity resolution is complete. This contradicts the initial idea to treat subsidaries as separate entities, but was added to improve the cleanliness of parameters and outputs for clustering.

**Input:** `contracts_cleaned.csv`  
**Output:** `contracts_resolved.csv`

In [53]:
import pandas as pd
import numpy as np
from datasketch import MinHash, MinHashLSH
import re
import roman

## Load Data

In [54]:
df = pd.read_csv('contracts_cleaned.csv', dtype={'naics_code': str})

print(f"Shape: {df.shape}")
print(f"Unique contractor names before resolution: {df['recipient_name'].nunique()}")

Shape: (73350, 6)
Unique contractor names before resolution: 24618


## Stage 2: MinHash Signatures

Each unique contractor name is converted into a set of character-level
3-shingles (overlapping 3-character substrings), then hashed into a MinHash
signature of 128 permutations. The MinHash signature allows efficient
approximation of Jaccard similarity between name strings without comparing
all pairs directly.

128 permutations gives a standard error of approximately ±0.088 on the
Jaccard estimate, sufficient precision for a threshold of 0.85.

In [55]:
# Generate character-level k-shingles from a string
def get_shingles(text, k=3):
    text = re.sub(r'\s+', '', text)  # remove spaces only
    return set(text[i:i+k] for i in range(len(text) - k + 1))

# Create a MinHash signature from a set of shingles
def make_minhash(shingles, num_perm=128):
    m = MinHash(num_perm=num_perm)
    for s in shingles:
        m.update(s.encode('utf-8'))
    return m

# Get unique names only: we only need one signature per unique string
unique_names = df['recipient_name'].unique()
print(f"Building MinHash signatures for {len(unique_names):,} unique names...")

signatures = {}
for name in unique_names:
    shingles = get_shingles(name)
    if len(shingles) > 0:
        signatures[name] = make_minhash(shingles)

print(f"Signatures built: {len(signatures):,}")

Building MinHash signatures for 24,618 unique names...
Signatures built: 24,618


## LSH Index & Candidate Pair Generation

MinHash signatures are inserted into a Locality Sensitive Hashing (LSH) index
configured at the same 0.85 threshold. LSH groups similar signatures into
buckets, allowing candidate pairs to be retrieved in sub-linear time rather
than requiring exhaustive pairwise comparison across all 24,618 names.

In [56]:
# Build LSH index
lsh = MinHashLSH(threshold=0.85, num_perm=128)

for name, sig in signatures.items():
    lsh.insert(name, sig)

# Query each name against the index to find similar pairs
print("Querying LSH index for similar name pairs...")

merge_pairs = []
seen = set()

for name, sig in signatures.items():
    results = lsh.query(sig)
    for match in results:
        if match != name:
            pair = tuple(sorted([name, match]))
            if pair not in seen:
                seen.add(pair)
                merge_pairs.append(pair)

print(f"Candidate pairs found: {len(merge_pairs)}")

Querying LSH index for similar name pairs...
Candidate pairs found: 655


## Candidate Filtering

655 candidate pairs are returned by LSH. Many are false positives: companies
sharing common generic suffixes like `SERVICES LLC` or `CONSTRUCTION INC` that
inflate Jaccard similarity despite being unrelated entities. Three sequential
filters are applied to eliminate false positives:

1. **First-word filter**: Both names must share the same first word. This
   eliminates pairs matched purely on shared generic suffixes.
2. **Number filter**: Arabic and Roman numerals must match exactly between
   both names. This preserves numbered joint ventures (e.g. `JV II` vs `JV III`)
   as distinct entities.
3. **Geographic qualifier filter**: If one name ends with a US state or
   territory that the other lacks, the pair is rejected. This prevents merging
   state-level subsidiaries (e.g. `WASTE MANAGEMENT OF VIRGINIA` vs
   `WASTE MANAGEMENT OF WEST VIRGINIA`).

In [57]:
US_STATES_AND_TERRITORIES = {
    'ALABAMA', 'ALASKA', 'ARIZONA', 'ARKANSAS', 'CALIFORNIA', 'COLORADO',
    'CONNECTICUT', 'DELAWARE', 'FLORIDA', 'GEORGIA', 'HAWAII', 'IDAHO',
    'ILLINOIS', 'INDIANA', 'IOWA', 'KANSAS', 'KENTUCKY', 'LOUISIANA',
    'MAINE', 'MARYLAND', 'MASSACHUSETTS', 'MICHIGAN', 'MINNESOTA',
    'MISSISSIPPI', 'MISSOURI', 'MONTANA', 'NEBRASKA', 'NEVADA',
    'NEW HAMPSHIRE', 'NEW JERSEY', 'NEW MEXICO', 'NEW YORK',
    'NORTH CAROLINA', 'NORTH DAKOTA', 'OHIO', 'OKLAHOMA', 'OREGON',
    'PENNSYLVANIA', 'RHODE ISLAND', 'SOUTH CAROLINA', 'SOUTH DAKOTA',
    'TENNESSEE', 'TEXAS', 'UTAH', 'VERMONT', 'VIRGINIA', 'WASHINGTON',
    'WEST VIRGINIA', 'WISCONSIN', 'WYOMING', 'GUAM', 'PUERTO RICO',
    'AMERICAN SAMOA'
}

def ends_with_geo_qualifier(name, other):
    """Returns True if name ends with a geographic qualifier
    the other name lacks — indicates geographic specialization,
    not a name variant."""
    for geo in US_STATES_AND_TERRITORIES:
        if name.endswith(geo) and not other.endswith(geo):
            return True
    return False

def first_word(name):
    tokens = name.split()
    return tokens[0] if tokens else ''

def extract_numbers(name):
    arabic = set(re.findall(r'\d+', name))
    roman_numerals = {'I', 'II', 'III', 'IV', 'V', 'VI', 'VII', 'VIII', 'IX', 'X'}
    tokens = set(name.split())
    return arabic | (tokens & roman_numerals)

filtered_pairs = []
for a, b in merge_pairs:
    if first_word(a) != first_word(b):
        continue
    if extract_numbers(a) != extract_numbers(b):
        continue
    if ends_with_geo_qualifier(a, b) or ends_with_geo_qualifier(b, a):
        continue
    filtered_pairs.append((a, b))

print(f"Pairs before filters: {len(merge_pairs)}")
print(f"After first-word filter: removed {len([p for p in merge_pairs if first_word(p[0]) != first_word(p[1])])}")
print(f"After numbers filter: removes numbered JV variants")
print(f"After geo filter: removes geographic specializations")
print(f"Final approved pairs: {len(filtered_pairs)}")

print("\n=== Filtered pairs ===\n")
for a, b in filtered_pairs:
    print(f"  '{a}'")
    print(f"  '{b}'")
    print()

Pairs before filters: 655
After first-word filter: removed 623
After numbers filter: removes numbered JV variants
After geo filter: removes geographic specializations
Final approved pairs: 11

=== Filtered pairs ===

  'HUNTSVILLE REHABILITATION FOUNDATION'
  'HUNTSVILLE REHABILITATION FOUNDATION INC'

  'MERIDIAN ENGINEERING CO'
  'MERIDIAN ENGINEERING INC'

  'WASTE MANAGEMENT OF VIRGINIA INC'
  'WASTE MANAGEMENT OF WEST VIRGINIA INC'

  'FEDERAL RESOURCES SUPPLY COMPANY'
  'FEDERAL RESOURCES SUPPLY COMPANY LLC'

  'UMASS MEMORIAL MEDICAL GROUP'
  'UMASS MEMORIAL MEDICAL GROUP INC'

  'L2EOCULUS JV'
  'L2EOCULUS JV LLC'

  'TYONEK SERVICES OVERHAUL FACILITY STENNIS LLC'
  'TYONEK SERVICES OVERHAUL FACILITYSTENNIS LLC'

  'VANDERBILT UNIVERSITY'
  'VANDERBILT UNIVERSITY THE'

  'SOUTHWESTERN BELL TELEPHONE COMPANY'
  'SOUTHWESTERN BELL TELEPHONE COMPANY LLC'

  'OMANG TECHNOLOGIES AND TRADING LLC'
  'OMANG TECHNOLOGIES AND TRADING LTD'

  'STERLING SHIPYARD LLC'
  'STERLING SHIPYARD L

## Canonical Name Mapping

A Union-Find (disjoint set) data structure handles transitive merges. If A
matches B and B matches C, all three are assigned the same canonical name.
The canonical name is chosen as the longest string in each group, representing
the most complete legal form of the name.

In [58]:
# Union-Find to handle transitive merges
parent = {name: name for name in unique_names}

def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x

def union(x, y):
    px, py = find(x), find(y)
    if px != py:
        # Canonical name = longest string (most complete form)
        if len(px) >= len(py):
            parent[py] = px
        else:
            parent[px] = py

# Apply all filtered pairs
for a, b in filtered_pairs:
    union(a, b)

# Build name -> canonical name mapping
name_to_canonical = {name: find(name) for name in unique_names}

# Summary
num_merged = sum(1 for k, v in name_to_canonical.items() if k != v)
print(f"Names that will be merged into a canonical form: {num_merged}")
print(f"Unique entities before resolution: {len(unique_names):,}")
print(f"Unique entities after resolution: {len(set(name_to_canonical.values())):,}")

# Show some merge groups
from collections import defaultdict
groups = defaultdict(list)
for name, canonical in name_to_canonical.items():
    if name != canonical:
        groups[canonical].append(name)

print(f"\n=== Sample merge groups ===")
for canonical, variants in list(groups.items())[:10]:
    print(f"\n  Canonical: '{canonical}'")
    for v in variants:
        print(f"    <- '{v}'")

Names that will be merged into a canonical form: 11
Unique entities before resolution: 24,618
Unique entities after resolution: 24,607

=== Sample merge groups ===

  Canonical: 'HUNTSVILLE REHABILITATION FOUNDATION INC'
    <- 'HUNTSVILLE REHABILITATION FOUNDATION'

  Canonical: 'FEDERAL RESOURCES SUPPLY COMPANY LLC'
    <- 'FEDERAL RESOURCES SUPPLY COMPANY'

  Canonical: 'MERIDIAN ENGINEERING INC'
    <- 'MERIDIAN ENGINEERING CO'

  Canonical: 'WASTE MANAGEMENT OF WEST VIRGINIA INC'
    <- 'WASTE MANAGEMENT OF VIRGINIA INC'

  Canonical: 'UMASS MEMORIAL MEDICAL GROUP INC'
    <- 'UMASS MEMORIAL MEDICAL GROUP'

  Canonical: 'L2EOCULUS JV LLC'
    <- 'L2EOCULUS JV'

  Canonical: 'TYONEK SERVICES OVERHAUL FACILITY STENNIS LLC'
    <- 'TYONEK SERVICES OVERHAUL FACILITYSTENNIS LLC'

  Canonical: 'VANDERBILT UNIVERSITY THE'
    <- 'VANDERBILT UNIVERSITY'

  Canonical: 'STERLING SHIPYARD LLC'
    <- 'STERLING SHIPYARD LP'

  Canonical: 'SOUTHWESTERN BELL TELEPHONE COMPANY LLC'
    <- 'SOUTH

## Apply & Save

In [59]:
# Apply canonical name mapping to full dataframe
df['recipient_name_original'] = df['recipient_name']
df['recipient_name'] = df['recipient_name'].map(name_to_canonical)

print(f"Unique contractor names before resolution: {df['recipient_name_original'].nunique():,}")
print(f"Unique contractor names after resolution: {df['recipient_name'].nunique():,}")
print(f"Null recipient_names after mapping: {df['recipient_name'].isna().sum()}")

# Save
df.to_csv('contracts_resolved.csv', index=False)
print("\nSaved contracts_resolved.csv")

Unique contractor names before resolution: 24,618
Unique contractor names after resolution: 24,607
Null recipient_names after mapping: 0

Saved contracts_resolved.csv
